# 01 - Insert MySQL data and read it with Spark JDBC

Create three dedicated MySQL tables, insert a small dataset and read it with Spark. This is a batch extraction exercise. No binlog, CDC reader or replication privileges are needed.

Use the existing Spark-enabled Python kernel. Run every cell in order. Keep the same configuration in all three notebooks. The `team` account needs CREATE, SELECT, INSERT and UPDATE in the existing `training` database.

## Configure the source
Credentials come from the existing environment or an interactive password prompt. Do not print JDBC_OPTIONS.

In [ ]:
import os
import re
import socket
from getpass import getpass

MYSQL_DB = "training"
TABLE_PREFIX = "lake16_"
LAB = "mysql_hdfs_2"
assert all(re.fullmatch(r"[a-z][a-z0-9_]*", value) for value in [MYSQL_DB, TABLE_PREFIX, LAB])
MYSQL_HOST = os.getenv("MYSQL_HOST", "mysql")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", "3306"))
MYSQL_USER = os.getenv("MYSQL_USER", os.getenv("DB_USER", "team"))
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD") or os.getenv("DB_PASSWORD") or getpass("MySQL password: ")
JDBC_URL = f"jdbc:mysql://{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}?useSSL=false&allowPublicKeyRetrieval=true&serverTimezone=UTC"
JDBC_OPTIONS = {"user": MYSQL_USER, "password": MYSQL_PASSWORD, "driver": "com.mysql.cj.jdbc.Driver", "fetchsize": "100"}
HDFS_URI = os.getenv("WORKSHOP_HDFS_URI", "hdfs://namenode:9000").rstrip("/")
assert HDFS_URI.startswith("hdfs://"), "Choose HDFS, not file:/// or S3."
HIVE_URI = os.getenv("HIVE_METASTORE_URI", "thrift://hive-metastore:9083")
HDFS_ROOT = f"{HDFS_URI}/workshop/16-mysql-hdfs-iceberg/{LAB}"
ORDERS_PATH = HDFS_ROOT + "/orders_parquet"
PRODUCTS_PATH = HDFS_ROOT + "/products_csv"
CUSTOMERS_PATH = HDFS_ROOT + "/customers_json"
HIVE_ORDERS = f"spark_catalog.{LAB}.orders_external_2"
LAKE_ROOT = f"s3://warehouse/iceberg/{LAB}"
print("MySQL source:", MYSQL_DB, "table prefix:", TABLE_PREFIX)
print("HDFS root:", HDFS_ROOT)

## Start Spark
The MySQL JDBC driver is already in the Spark image. No Python MySQL client or helper file is required.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pathlib import Path
assert SparkSession.getActiveSession() is None, "Stop the previous Spark session and restart this kernel."
jars = Path(os.getenv("SPARK_HOME", "/opt/spark")) / "jars"
assert list(jars.glob("mysql-connector-j-*.jar")), "Use the Spark image with its MySQL JDBC driver."
builder = (SparkSession.builder.appName("mysql-hdfs-" + LAB)
    .master(os.getenv("SPARK_MASTER_URL", "spark://spark-master:7077"))
    .config("spark.driver.host", os.getenv("POD_IP") or socket.gethostbyname(socket.gethostname()))
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.cores.max", "1").config("spark.executor.cores", "1")
    .config("spark.executor.memory", "512m")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.redaction.regex", "(?i)secret|password|token|access.?key|credential")
    .config("spark.hadoop.fs.defaultFS", "file:///")
    .config("spark.hadoop.dfs.client.use.datanode.hostname", "true")
    .config("spark.hadoop.dfs.replication", "1"))

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version, "master:", spark.sparkContext.master)

## Create isolated source tables

The names are `lake16_products_2`, `lake16_customers_2` and `lake16_orders_2` in MySQL database `training`. They are separate from the SCD exercise. Repeating the seed updates these same primary keys instead of inserting duplicates.

The driver-side JDBC connection below executes DDL and inserts. The following section uses Spark's JDBC reader to distribute extraction. Database DDL commits independently of the insert transaction.

In [ ]:
j = spark.sparkContext._jvm
connection_properties = j.java.util.Properties()
connection_properties.setProperty("user", MYSQL_USER)
connection_properties.setProperty("password", MYSQL_PASSWORD)
connection = j.com.mysql.cj.jdbc.Driver().connect(JDBC_URL, connection_properties)
try:
    statement = connection.createStatement()
    try:
        statement.executeUpdate(f"""CREATE TABLE IF NOT EXISTS {TABLE_PREFIX}products_2 (
            product_id BIGINT PRIMARY KEY, product_name VARCHAR(100) NOT NULL,
            category VARCHAR(40), list_price DECIMAL(12,2) NOT NULL)""")
        statement.executeUpdate(f"""CREATE TABLE IF NOT EXISTS {TABLE_PREFIX}customers_2 (
            customer_id BIGINT PRIMARY KEY, customer_name VARCHAR(80) NOT NULL,
            country VARCHAR(2) NOT NULL, email VARCHAR(100))""")
        statement.executeUpdate(f"""CREATE TABLE IF NOT EXISTS {TABLE_PREFIX}orders_2 (
            order_id BIGINT PRIMARY KEY, customer_id BIGINT NOT NULL, product_id BIGINT NOT NULL,
            quantity INT NOT NULL, unit_price DECIMAL(12,2) NOT NULL, country VARCHAR(2) NOT NULL)""")
        connection.setAutoCommit(False)
        statement.executeUpdate(f"""INSERT INTO {TABLE_PREFIX}products_2 VALUES
            (101,'Keyboard, Compact','Accessories',25.00),
            (102,'Wireless Mouse','Accessories',15.00),
            (103,'USB Cable','Connectivity',10.00),
            (104,'Docking Station',NULL,50.00)
            ON DUPLICATE KEY UPDATE product_name=VALUES(product_name), category=VALUES(category), list_price=VALUES(list_price)""")
        statement.executeUpdate(f"""INSERT INTO {TABLE_PREFIX}customers_2 VALUES
            (1,'Asha','IN','asha@example.test'),(2,'Ravi','IN',NULL),
            (3,'Emma','UK','emma@example.test'),(4,'Liam','US','liam@example.test')
            ON DUPLICATE KEY UPDATE customer_name=VALUES(customer_name), country=VALUES(country), email=VALUES(email)""")
        statement.executeUpdate(f"""INSERT INTO {TABLE_PREFIX}orders_2 VALUES
            (2001,1,101,2,25.00,'IN'),(2002,2,102,1,15.00,'IN'),
            (2003,3,103,3,10.00,'UK'),(2004,4,104,1,50.00,'US'),
            (2005,1,103,2,10.00,'IN'),(2006,3,101,1,25.00,'UK')
            ON DUPLICATE KEY UPDATE customer_id=VALUES(customer_id), product_id=VALUES(product_id),
              quantity=VALUES(quantity), unit_price=VALUES(unit_price), country=VALUES(country)""")
        connection.commit()
    except Exception:
        connection.rollback()
        raise
    finally:
        statement.close()
finally:
    connection.close()
print("PASS: MySQL tables created and seed committed")

## Read MySQL using Spark JDBC

Products and customers use `spark.read.jdbc`. Orders use the equivalent format/options API and two JDBC partitions. Lower and upper bounds calculate partition strides; they do not filter records. Keep MySQL unchanged while reading: separate connections do not provide a coordinated source snapshot.

In [ ]:
# These are full table reads, not CDC or incremental extraction.
products = spark.read.jdbc(JDBC_URL, TABLE_PREFIX + "products_2", properties=JDBC_OPTIONS)
customers = spark.read.jdbc(JDBC_URL, TABLE_PREFIX + "customers_2", properties=JDBC_OPTIONS)
orders = (spark.read.format("jdbc").option("url", JDBC_URL)
    .option("dbtable", TABLE_PREFIX + "orders_2")
    .options(**JDBC_OPTIONS)
    .option("partitionColumn", "order_id").option("lowerBound", "2001")
    .option("upperBound", "2007").option("numPartitions", "2").load())
assert products.count() == 4 and customers.count() == 4 and orders.count() == 6
assert orders.select(F.sum(F.col("quantity") * F.col("unit_price"))).first()[0] == 190
for name, frame, key in [("products", products, "product_id"), ("customers", customers, "customer_id"), ("orders", orders, "order_id")]:
    print(name)
    frame.orderBy(key).show(truncate=False)
print("PASS: JDBC source counts and order value 190.00")

## Check data types and keys

DECIMAL represents money exactly. Primary keys identify business records. The null product category and customer email exercise null handling; the comma in a product name exercises CSV quoting in notebook 02.

In [ ]:
for frame, key in [(products,"product_id"),(customers,"customer_id"),(orders,"order_id")]:
    frame.printSchema()
    assert frame.select(key).distinct().count() == frame.count()
assert products.filter("category IS NULL").count() == 1
assert customers.filter("email IS NULL").count() == 1
print("PASS: unique keys, decimal prices and nullable attributes")

## Release Spark
Run this cell before starting the next notebook. Use a fresh kernel for the next notebook.

In [ ]:
spark.stop()